# MVP Report Chat (Retrieval-Only)

This notebook builds the report index and provides a simple chat UI with current-report-first retrieval and cross-report fallback.

In [1]:
%pip install --upgrade pip setuptools wheel
%pip install lxml beautifulsoup4 ipywidgets

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.0
    Uninstalling setuptools-82.0.0:
      Successfully uninstalled setuptools-82.0.0
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 35.2 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys
import sqlite3

try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    raise RuntimeError("Please install ipywidgets: pip install ipywidgets") from exc

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.indexer import build_index
from src.chat import chat_turn


In [3]:
JSON_DIR = ROOT / '_reports_json'
HTML_DIR = ROOT / '_reports_html'
DB_PATH = ROOT / 'data' / 'reports_index.db'

build_index(str(JSON_DIR), str(HTML_DIR), str(DB_PATH))

conn = sqlite3.connect(str(DB_PATH))
doc_ids = [r[0] for r in conn.execute('SELECT doc_id FROM documents ORDER BY doc_id')]
conn.close()

print(f'Index built: {DB_PATH}')
print(f'Documents indexed: {len(doc_ids)}')
print('Sample docs:', doc_ids[:8])

Index built: /Users/atheeshkrishnan/AK/DEV/infoapp-dynamic-retrieval/data/reports_index.db
Documents indexed: 6
Sample docs: ['bar-only.html', 'bar-only.json', 'credit_report.html', 'credit_report.json', 'piechart.html', 'piechart.json']


In [4]:
report_selector = widgets.Dropdown(
    options=doc_ids,
    value='bar-only.json' if 'bar-only.json' in doc_ids else (doc_ids[0] if doc_ids else None),
    description='Report:',
    layout=widgets.Layout(width='420px')
)

question_box = widgets.Textarea(
    placeholder='Ask about this report...',
    layout=widgets.Layout(width='720px', height='90px')
)

send_button = widgets.Button(description='Send', button_style='primary')
clear_button = widgets.Button(description='Clear', button_style='warning')

chat_html = widgets.HTML(value='<b>Chat</b><hr>')
evidence_accordion = widgets.Accordion(children=[widgets.HTML(value='No evidence yet.')])
evidence_accordion.set_title(0, 'Evidence')

history = []

def render_history():
    parts = ["<b>Chat</b><hr>"]
    for role, text in history:
        safe = (
            text.replace('&', '&amp;')
            .replace('<', '&lt;')
            .replace('>', '&gt;')
            .replace('\n', '<br>')
        )
        color = '#1f2937' if role == 'You' else '#111827'
        parts.append(
            f"<div style='margin:8px 0;padding:8px 10px;border-radius:10px;background:{color};color:white;'>"
            f"<b>{role}:</b><br>{safe}</div>"
        )
    chat_html.value = ''.join(parts)

def render_evidence(evidence):
    if not evidence:
        evidence_accordion.children = [widgets.HTML(value='No evidence.')]
        evidence_accordion.set_title(0, 'Evidence')
        return

    rows = []
    for idx, item in enumerate(evidence[:8], start=1):
        row_num = item.get('row_num')
        row_txt = f" | row={row_num}" if row_num is not None else ''
        rows.append(
            f"<div style='margin:6px 0;padding:6px;border:1px solid #ddd;border-radius:8px;'>"
            f"<b>{idx}. {item['doc_id']}</b> | {item['chunk_type']}{row_txt}<br>"
            f"<i>{item['source_path']}</i><br>"
            f"{item['snippet']}<br>"
            f"<span style='opacity:.7'>score={item['score']}</span>"
            f"</div>"
        )

    evidence_accordion.children = [widgets.HTML(value=''.join(rows))]
    evidence_accordion.set_title(0, 'Evidence')

def on_send(_):
    query = question_box.value.strip()
    if not query:
        return
    current_doc_id = report_selector.value

    history.append(('You', query))
    result = chat_turn(query=query, current_doc_id=current_doc_id, db_path=str(DB_PATH))
    history.append(('Agent', result['answer']))

    render_history()
    render_evidence(result['evidence'])
    question_box.value = ''

def on_clear(_):
    history.clear()
    render_history()
    render_evidence([])

send_button.on_click(on_send)
clear_button.on_click(on_clear)

controls = widgets.HBox([send_button, clear_button])
ui = widgets.VBox([report_selector, question_box, controls, chat_html, evidence_accordion])

render_history()
render_evidence([])
display(ui)